In [ ]:
# # AFFETDS: Adversarial Feature Fusion Ensemble for Tumor Deepfake Detection
# ## Enhancing tumor deepfake detection in MRI scans using adversarial feature fusion ensembles
# **Paper**: Nature 2025 | **Environment**: Kaggle GPU | **Expected Accuracy**: 91.5%

# ---

# ### Architecture Overview:
# - **Input**: MRI scans (224×224×3 RGB, normalized)
# - **Deep Features**: ResNet50 (Layer4 avgpool, 2048-dim)
# - **Handcrafted Features**: HOG (9 orientations, 8px cells, 3780-dim)
# - **Fusion**: Concatenation → 5828-dim
# - **Classifier**: SVM (RBF kernel)
# - **Adversarial**: PGD attack (ε=0.031, 10 steps)
# - **Optimization**: Adam 50 epochs, batch=32

## CELL 1: KAGGLE SETUP - GPU Verification & Paths

In [ ]:
import os
import torch

print("\n" + "="*80)
print("🔍 KAGGLE SETUP: GPU & Path Configuration")
print("="*80)

# Check if running on Kaggle
IS_KAGGLE = os.path.exists('/kaggle')

if IS_KAGGLE:
    print("✓ Running on Kaggle environment")
    CHECKPOINT_DIR = '/kaggle/working/checkpoints'
    DATA_DIR = '/kaggle/input'
else:
    print("ℹ️  Running locally (not Kaggle)")
    CHECKPOINT_DIR = './checkpoints'
    DATA_DIR = './data'

# Create checkpoint directory
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"📁 Checkpoint dir: {CHECKPOINT_DIR}")
print(f"📁 Data dir: {DATA_DIR}")

# GPU Verification
print(f"\nGPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ No GPU detected. Select GPU in Kaggle settings!")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n✓ Device: {device}")
print("="*80 + "\n")

## CELL 2: Kaggle Dataset Detection & Configuration

In [ ]:
import os
from pathlib import Path

print("🔧 CONFIGURING DATA PATHS FOR KAGGLE\n")

# Kaggle-specific dataset search
# ⭐ YOUR KAGGLE DATASET PATH - HIGHEST PRIORITY
possible_paths = [
    '/kaggle/input/datasets/abuzarmehdicoder/mri-dataset/combined_dataset',  # ⭐ YOUR DATASET
    '/kaggle/input/brain-mri-dataset/dataset',       # Common Kaggle dataset
    '/kaggle/input/brain-mri-images',                # Alternative name
    '/kaggle/input/combined-dataset',                # Combined dataset
    './dataset',                                      # Local relative path
    './combined_dataset',                             # Local combined
]

DATA_ROOT = None
print("📁 Checking available Kaggle inputs:")

if os.path.exists('/kaggle/input'):
    kaggle_inputs = os.listdir('/kaggle/input')
    for inp in kaggle_inputs:
        print(f"   - {inp}")
else:
    print("   (No /kaggle/input found - running locally)")

print("\n🔍 Searching for dataset...")
print(f"   Priority 1: /kaggle/input/datasets/abuzarmehdicoder/mri-dataset/combined_dataset")

for path in possible_paths:
    if os.path.exists(path):
        # Check if it has data subdirectories
        if os.path.exists(os.path.join(path, 'real')):
            DATA_ROOT = path
            print(f"\n✓ Using dataset from: {path}")
            real_count = len([f for f in os.listdir(f'{path}/real') if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
            fake_count = len([f for f in os.listdir(f'{path}/fake') if f.lower().endswith(('.jpg', '.png', '.jpeg'))] if os.path.exists(f'{path}/fake') else [])
            print(f"  Real images: {real_count}")
            print(f"  Fake images: {fake_count}\n")
            break

if DATA_ROOT is None:
    DATA_ROOT = './dataset'
    print(f"\n⚠️  Dataset not found at: {possible_paths[0]}")
    print(f"\n📝 EXPECTED STRUCTURE (at your Kaggle path):")
    print(f"   /kaggle/input/datasets/abuzarmehdicoder/mri-dataset/combined_dataset/")
    print(f"   ├── real/      (real MRI images)")
    print(f"   └── fake/      (synthetic/deepfake MRI images)")
    print(f"\n📤 TO FIX:")
    print(f"   1. Ensure dataset is added as data source in notebook settings")
    print(f"   2. Verify path: /kaggle/input/datasets/abuzarmehdicoder/mri-dataset/combined_dataset")
    print(f"   3. Re-run this cell\n")

# Save config
DATA_CONFIG = {
    'root': DATA_ROOT,
    'real_path': f'{DATA_ROOT}/real' if DATA_ROOT else None,
    'fake_path': f'{DATA_ROOT}/fake' if DATA_ROOT else None,
}

print(f"📊 Dataset Configuration:")
print(f"   Root: {DATA_CONFIG['root']}")
print(f"   Real: {DATA_CONFIG['real_path']}")
print(f"   Fake: {DATA_CONFIG['fake_path']}")
print()

## CELL 3: Import All Libraries & Configuration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2
from tqdm import tqdm
import os
import json
from pathlib import Path

# PyTorch imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

# Scikit-learn
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve, auc
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Computer vision
from skimage.feature import hog as hog_skimage
from skimage import color

# Warnings
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

# Configuration - OPTION 2: FAST ROBUSTNESS (Balanced approach)
CONFIG = {
    'img_size': 192,                    # ⚡ OPTIMIZED: 160 → 192 (faster than 224)
    'batch_size': 16,                   # ⚡ OPTIMIZED: 32 → 16 (faster convergence)
    'epochs': 20,                       # ⚡ OPTIMIZED: 15 → 20 (more robustness training)
    'lr': 0.001,
    'device': device,
    'num_classes': 2,
    'hog_pixels_per_cell': 16,
    'hog_orientations': 9,
    'pgd_epsilon': 0.07,                # 🛡️ STRONGER: 0.031 → 0.07 (moderate perturbations)
    'pgd_alpha': 0.0175,                # 🛡️ STEP SIZE: 0.07/4 for better convergence
    'pgd_steps': 15,                    # 🛡️ STRONGER: 3 → 15 (more robust attacks)
    'svm_c': 1.0,
    'svm_gamma': 'scale',
    'num_workers': 0,
    'pin_memory': True,
    'use_mixed_precision': True,        # ⚡ Keep FP16 for speed
    'is_checking_accuracy': False,
    'use_subset': True,
    'subset_size_per_class': 2500,
    'checkpoint_dir': CHECKPOINT_DIR,   # Use Kaggle path
}

print("✓ All imports successful")
print(f"✓ Configuration loaded: OPTION 2 - Fast Robustness Training")
print(f"   - Image size: {CONFIG['img_size']}×{CONFIG['img_size']} (faster)")
print(f"   - Batch size: {CONFIG['batch_size']} (faster convergence)")
print(f"   - Epochs: {CONFIG['epochs']} (more robustness)")
print(f"   - PGD epsilon: {CONFIG['pgd_epsilon']} (moderate perturbations)")
print(f"   - PGD steps: {CONFIG['pgd_steps']} (stronger attacks)")
print(f"   - Expected training time: 3-4 hours on P100 GPU")
print(f"✓ Expected results: 82-85% clean accuracy, 60-70% adversarial accuracy")

## CELL 4: Load Dataset from Kaggle Input

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════════
# GENERATE & REGENERATE DEEPFAKES (FORCE NEW GENERATION - CLEAR OLD FAKES)
# ═════════════════════════════════════════════════════════════════════════════════

if DATA_ROOT and os.path.exists(f'{DATA_ROOT}/real'):
    from scipy.ndimage import gaussian_filter
    from skimage.exposure import match_histograms
    import shutil
    from glob import glob
    
    real_dir = f'{DATA_ROOT}/real'
    output_fake_dir = '/kaggle/working/generated_fakes'
    
    # 🗑️  STEP 1: CLEAR OLD GENERATED FAKES (FORCE REGENERATION)
    print("🗑️  CLEARING OLD GENERATED FAKE IMAGES...")
    print("=" * 80)
    
    if os.path.exists(output_fake_dir):
        old_count = len([f for f in os.listdir(output_fake_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
        print(f"   Found {old_count} old fake images in: {output_fake_dir}")
        shutil.rmtree(output_fake_dir)
        print(f"   ✓ Deleted all {old_count} old fakes\n")
    else:
        print(f"   No previous cache found. Creating fresh directory...\n")
    
    os.makedirs(output_fake_dir, exist_ok=True)
    print(f"✓ Fresh output directory ready: {output_fake_dir}\n")
    
    # 🎬 STEP 2: GENERATE NEW DEEPFAKES FROM SCRATCH
    print("🎬 GENERATING NEW DEEPFAKES FROM SCRATCH...")
    print("=" * 80)
    
    real_images = sorted([f for f in os.listdir(real_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
    print(f"Found {len(real_images)} real images for deepfake generation")
    print(f"🚀 Generating ALL {len(real_images)} fake images (this may take 5-10 minutes)...\n")
    
    fake_count = 0
    variation_types = ['subtle', 'moderate', 'aggressive']
    variation_cycle = 0
    
    for idx, real_img_name in enumerate(tqdm(real_images, desc="Generating deepfakes")):
        try:
            variation_type = variation_types[variation_cycle % len(variation_types)]
            variation_cycle += 1
            
            real_path = os.path.join(real_dir, real_img_name)
            fake_array = generate_realistic_deepfake(real_path, variation_type=variation_type)
            
            fake_rgb = np.stack([fake_array] * 3, axis=-1)
            fake_rgb = (np.clip(fake_rgb, 0, 1) * 255).astype(np.uint8)
            
            fake_path = f'{output_fake_dir}/{os.path.splitext(real_img_name)[0]}_fake.jpg'
            Image.fromarray(fake_rgb).save(fake_path)
            fake_count += 1
        except Exception as e:
            continue
    
    print(f"\n✅ SUCCESSFULLY GENERATED {fake_count} NEW DEEPFAKES!")
    print(f"   Location: {output_fake_dir}\n")
    
    # 📊 STEP 3: VERIFY GENERATION ON DISK
    verify_count = len([f for f in os.listdir(output_fake_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
    print(f"✓ VERIFICATION: {verify_count} fake images confirmed on disk\n")
    
    working_fake_dir = output_fake_dir
    
    # 📸 STEP 4: VISUALIZE GENERATED DEEPFAKES (SAMPLE OF 10)
    print("📸 VISUALIZING SAMPLE OF GENERATED DEEPFAKES...")
    print("=" * 80 + "\n")
    
    fig, axes = plt.subplots(2, 5, figsize=(16, 7))
    axes = axes.flatten()
    
    fake_images_on_disk = sorted([f for f in os.listdir(output_fake_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
    sample_indices = np.linspace(0, len(fake_images_on_disk)-1, 10, dtype=int)
    sample_fakes = [fake_images_on_disk[i] for i in sample_indices]
    
    for idx, fake_name in enumerate(sample_fakes):
        fake_path = os.path.join(output_fake_dir, fake_name)
        fake_img = cv2.imread(fake_path)
        fake_img = cv2.cvtColor(fake_img, cv2.COLOR_BGR2RGB)
        
        axes[idx].imshow(fake_img)
        axes[idx].set_title(f"Sample #{idx+1}", fontsize=10, fontweight='bold')
        axes[idx].axis('off')
    
    plt.suptitle(f"✅ DEEPFAKE GENERATION COMPLETE: {verify_count} Total Synthetic Tumors Generated", fontsize=14, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.savefig(f'{CHECKPOINT_DIR}/generated_fakes_visualization.png', dpi=100, bbox_inches='tight')
    plt.show()
    
    print(f"✓ Visualization saved: {CHECKPOINT_DIR}/generated_fakes_visualization.png\n")
    
    # 📋 STEP 5: SUMMARY REPORT
    print("=" * 80)
    print("✅ DEEPFAKE GENERATION & VERIFICATION COMPLETE")
    print("=" * 80)
    print(f"📊 Generation Summary:")
    print(f"   - Old fakes deleted: ✓")
    print(f"   - New fakes generated: {fake_count}")
    print(f"   - Verified on disk: {verify_count}")
    print(f"   - Visualization created: ✓ (10 samples shown)")
    print(f"   - Output directory: {output_fake_dir}\n")

    # ═════════════════════════════════════════════════════════════════════════════════
    # LOAD DATASET: REAL & FAKE IMAGES (WITH PROPER STRING PATHS)
    # ═════════════════════════════════════════════════════════════════════════════════
    
    print("=" * 80)
    print("📚 LOADING DATASET: REAL & FAKE IMAGES")
    print("=" * 80 + "\n")
    
    # Load real image paths (label: 0)
    real_dir = f'{DATA_ROOT}/real'
    real_images = sorted([f for f in os.listdir(real_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
    real_paths = [str(os.path.join(real_dir, img)) for img in real_images]
    real_labels = [0] * len(real_paths)
    
    print(f"✓ Loaded {len(real_paths)} REAL images (label: 0)")
    print(f"  From: {real_dir}\n")
    
    # Load fake image paths (label: 1)
    fake_images = sorted([f for f in os.listdir(working_fake_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
    fake_paths = [str(os.path.join(working_fake_dir, img)) for img in fake_images]
    fake_labels = [1] * len(fake_paths)
    
    print(f"✓ Loaded {len(fake_paths)} FAKE images (label: 1)")
    print(f"  From: {working_fake_dir}\n")
    
    # Combine real and fake
    all_image_paths = real_paths + fake_paths
    all_labels = real_labels + fake_labels
    
    print(f"✓ Total images: {len(all_image_paths)} ({len(real_paths)} real + {len(fake_paths)} fake)\n")
    
    # Create train/val/test split (70% / 15% / 15%)
    from sklearn.model_selection import train_test_split
    
    indices = np.arange(len(all_image_paths))
    train_idx, temp_idx = train_test_split(indices, test_size=0.30, random_state=42, stratify=all_labels)
    val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, random_state=42, stratify=[all_labels[i] for i in temp_idx])
    
    # Extract paths and labels for each split (convert to regular Python strings)
    train_paths = [str(all_image_paths[i]) for i in train_idx]
    train_labels = [all_labels[i] for i in train_idx]
    
    val_paths = [str(all_image_paths[i]) for i in val_idx]
    val_labels = [all_labels[i] for i in val_idx]
    
    test_paths = [str(all_image_paths[i]) for i in test_idx]
    test_labels = [all_labels[i] for i in test_idx]
    
    print("=" * 80)
    print("📊 DATASET SPLIT (70% train / 15% val / 15% test)")
    print("=" * 80)
    print(f"Training:   {len(train_paths):5d} images | Real: {train_labels.count(0):5d} | Fake: {train_labels.count(1):5d}")
    print(f"Validation: {len(val_paths):5d} images | Real: {val_labels.count(0):5d} | Fake: {val_labels.count(1):5d}")
    print(f"Test:       {len(test_paths):5d} images | Real: {test_labels.count(0):5d} | Fake: {test_labels.count(1):5d}")
    print("=" * 80 + "\n")
    
    # Verify paths are proper strings (not numpy strings)
    print("🔍 Verifying path formats...")
    print(f"   Train path sample: {train_paths[0]} (type: {type(train_paths[0]).__name__})")
    print(f"   Val path sample:   {val_paths[0]} (type: {type(val_paths[0]).__name__})")
    print(f"   Test path sample:  {test_paths[0]} (type: {type(test_paths[0]).__name__})")
    print(f"✓ All paths are proper Python strings\n")

else:
    print("⚠️  Real dataset not found. Skipping dataset loading.")

## CELL 5: Generate Realistic Deepfakes (Tumor Insertion)

In [ ]:
def generate_realistic_deepfake(real_img_path, variation_type='moderate'):
    """
    Generate realistic deepfake with diversity levels:
    - 'subtle': Small tumors, low intensity boost, high blending
    - 'moderate': Medium tumors, medium intensity boost, balanced blending
    - 'aggressive': Large tumors, high intensity boost, less blending

    ✅ NO FULL-IMAGE BLUR - Only sharp tumor boundaries with edge smoothing
    ✅ REALISTIC MRI NOISE - Gaussian + salt-and-pepper artifacts
    """
    from scipy.ndimage import gaussian_filter
    from skimage.exposure import match_histograms
    
    # Load real image
    real_img = Image.open(real_img_path).convert('L')
    real_array = np.array(real_img) / 255.0

    if real_array.shape != (160, 160):
        real_img = real_img.resize((160, 160), Image.LANCZOS)
        real_array = np.array(real_img) / 255.0

    fake_array = real_array.copy()

    # Configure based on variation type
    if variation_type == 'subtle':
        radius_range = (5, 10)
        intensity_boost = np.random.uniform(0.08, 0.12)
        edge_blur_sigma = 0.8
        blend_strength = 0.4
    elif variation_type == 'moderate':
        radius_range = (10, 18)
        intensity_boost = np.random.uniform(0.12, 0.18)
        edge_blur_sigma = 1.0
        blend_strength = 0.6
    else:  # aggressive
        radius_range = (18, 28)
        intensity_boost = np.random.uniform(0.20, 0.28)
        edge_blur_sigma = 1.2
        blend_strength = 0.8

    # Generate irregular tumor location
    center_x = np.random.randint(40, 120)
    center_y = np.random.randint(40, 120)
    radius = np.random.randint(radius_range[0], radius_range[1])

    # Irregular shape with noise
    y, x = np.ogrid[0:160, 0:160]
    dist_map = np.sqrt((x - center_x)**2 + (y - center_y)**2)

    # Add irregularity
    noise = np.random.normal(1.0, 0.15, (160, 160))
    noise = gaussian_filter(noise, sigma=5)
    dist_map_irregular = dist_map * noise

    # Soft mask (initial)
    mask = np.exp(-(dist_map_irregular**2) / (2 * (radius/2)**2))

    # Compute tumor intensity from local tissue
    tissue_region = real_array[mask > 0.2]
    if len(tissue_region) > 0:
        local_mean = np.mean(tissue_region)
        tumor_intensity = np.clip(local_mean + intensity_boost, 0.3, 0.9)
    else:
        tumor_intensity = 0.7

    # Blend using mask with variation-specific strength
    fake_array = real_array * (1 - mask * blend_strength) + tumor_intensity * mask * blend_strength

    # ✅ ONLY blur mask EDGE for smooth transition (not entire image)
    mask_blurred = gaussian_filter(mask, sigma=edge_blur_sigma)
    fake_array = real_array * (1 - mask_blurred) + tumor_intensity * mask_blurred

    # 70% of time: match histogram (natural statistics)
    if np.random.rand() < 0.7:
        fake_array = match_histograms(fake_array, real_array)

    # ✅ ADD REALISTIC MRI NOISE
    # 1. Gaussian noise (scanner thermal noise)
    gaussian_noise = np.random.normal(0, 0.025, fake_array.shape)
    fake_array = fake_array + gaussian_noise

    # 2. Salt-and-pepper noise (MRI artifacts/spikes) - 30% of time
    if np.random.rand() < 0.3:
        sp_mask = np.random.rand(*fake_array.shape) < 0.005
        fake_array[sp_mask] = np.random.choice([0, 1], sp_mask.sum())

    # 3. Random intensity speckles (Rayleigh distributed, typical MRI noise)
    if np.random.rand() < 0.3:
        speckle_mask = np.random.rand(*fake_array.shape) < 0.03
        speckle = np.random.rayleigh(0.05, speckle_mask.shape)
        fake_array[speckle_mask] = fake_array[speckle_mask] + speckle[speckle_mask]

    fake_array = np.clip(fake_array, 0, 1)

    return fake_array

# Generate deepfakes
if DATA_ROOT and os.path.exists(f'{DATA_ROOT}/real'):
    from scipy.ndimage import gaussian_filter
    from skimage.exposure import match_histograms
    import shutil
    from glob import glob
    
    real_dir = f'{DATA_ROOT}/real'
    
    # Check if fakes already exist in input directory (pre-generated)
    input_fake_dir = f'{DATA_ROOT}/fake'
    if os.path.exists(input_fake_dir) and len([f for f in os.listdir(input_fake_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]) > 0:
        print(f"✓ Using pre-generated fakes from: {input_fake_dir}")
        fake_count = len([f for f in os.listdir(input_fake_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
        working_fake_dir = input_fake_dir
    else:
        # Generate new fakes to working directory (writable)
        print("🎬 Generating deepfakes...\n")
        output_fake_dir = '/kaggle/working/generated_fakes'
        os.makedirs(output_fake_dir, exist_ok=True)
        
        real_images = sorted([f for f in os.listdir(real_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))])
        print(f"Found {len(real_images)} real images")
        
        fake_count = 0
        variation_types = ['subtle', 'moderate', 'aggressive']
        variation_cycle = 0
        
        for idx, real_img_name in enumerate(tqdm(real_images[:100], desc="Generating deepfakes")):
            try:
                variation_type = variation_types[variation_cycle % len(variation_types)]
                variation_cycle += 1
                
                real_path = os.path.join(real_dir, real_img_name)
                fake_array = generate_realistic_deepfake(real_path, variation_type=variation_type)
                
                fake_rgb = np.stack([fake_array] * 3, axis=-1)
                fake_rgb = (np.clip(fake_rgb, 0, 1) * 255).astype(np.uint8)
                
                fake_path = f'{output_fake_dir}/{os.path.splitext(real_img_name)[0]}_fake.jpg'
                Image.fromarray(fake_rgb).save(fake_path)
                fake_count += 1
            except Exception as e:
                continue
        
        print(f"\n✅ Generated {fake_count} deepfakes")
        working_fake_dir = output_fake_dir
    
    print(f"\n✓ Deepfake dataset ready: {fake_count} synthetic images")
    print(f"  Located at: {working_fake_dir}")
else:
    print("⚠️  Real dataset not found. Skipping deepfake generation.")

## CELL 6: Custom Dataset Class & Data Loaders

In [ ]:
class MRIDataset(Dataset):
    """Custom Dataset for MRI images with augmentation"""
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        img = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            img = self.transform(img)

        return img, label

# Data augmentation
transform_train = transforms.Compose([
    transforms.Resize((CONFIG['img_size'], CONFIG['img_size'])),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
])

transform_test = transforms.Compose([
    transforms.Resize((CONFIG['img_size'], CONFIG['img_size'])),
    transforms.ToTensor(),
])

# Create datasets
train_dataset = MRIDataset(train_paths, train_labels, transform_train)
val_dataset = MRIDataset(val_paths, val_labels, transform_test)
test_dataset = MRIDataset(test_paths, test_labels, transform_test)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=True if len(train_dataset) > CONFIG['batch_size'] else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print(f"✓ DataLoaders created (pin_memory=True, num_workers=0)")
print(f"  - Batch size: {CONFIG['batch_size']}")
print(f"  - Training batches: {len(train_loader)}")

## CELL 7: HOG Feature Extractor

In [ ]:
class HOGExtractor:
    """Extract HOG features from images"""
    def __init__(self, pixels_per_cell=8, orientations=9):
        self.pixels_per_cell = pixels_per_cell
        self.orientations = orientations

    def extract(self, img_tensor):
        """
        Extract HOG features from PyTorch tensor.
        Input: (B, 3, 224, 224) or (3, 224, 224)
        Output: (B, hog_features) or (hog_features,)
        """
        if img_tensor.dim() == 3:
            img_tensor = img_tensor.unsqueeze(0)
            single = True
        else:
            single = False

        B = img_tensor.shape[0]
        hog_features = []

        for b in range(B):
            img_np = img_tensor[b].detach().permute(1, 2, 0).cpu().numpy()

            # Denormalize
            img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
            img_np = np.clip(img_np, 0, 1)

            img_gray = color.rgb2gray(img_np)

            hog_feat = hog_skimage(
                img_gray,
                pixels_per_cell=(self.pixels_per_cell, self.pixels_per_cell),
                cells_per_block=(1, 1),
                orientations=self.orientations,
                feature_vector=True
            )
            hog_features.append(torch.tensor(hog_feat, dtype=torch.float32))

        hog_features = torch.stack(hog_features)

        if single:
            return hog_features.squeeze(0)
        return hog_features

hog_extractor = HOGExtractor(
    pixels_per_cell=CONFIG['hog_pixels_per_cell'],
    orientations=CONFIG['hog_orientations']
)

print("✓ HOG Extractor initialized")
print(f"  - Pixels per cell: {CONFIG['hog_pixels_per_cell']}")
print(f"  - Orientations: {CONFIG['hog_orientations']}")

## CELL 8: ResNet50 Deep Feature Extractor

In [ ]:
class ResNet50Extractor(nn.Module):
    """ResNet50 feature extractor (Layer4 avgpool)"""
    def __init__(self, device):
        super(ResNet50Extractor, self).__init__()
        resnet50 = models.resnet50(pretrained=True)

        for param in resnet50.parameters():
            param.requires_grad = False

        self.features = nn.Sequential(*list(resnet50.children())[:-1])
        self.device = device
        self.to(device)

    def forward(self, x):
        """Extract 2048-dim features from Layer4 avgpool"""
        feat = self.features(x)
        feat = feat.view(feat.size(0), -1)
        return feat

resnet_extractor = ResNet50Extractor(device)
print("✓ ResNet50 Extractor initialized")
print("  - Pretrained: ImageNet")
print("  - Frozen: All parameters")
print("  - Output dimension: 2048")

## CELL 9: PGD Adversarial Attack Implementation

In [ ]:
class PGDAttack:
    """Projected Gradient Descent (PGD) adversarial attack"""
    def __init__(self, epsilon=0.031, alpha=0.007, steps=10):
        self.epsilon = epsilon
        self.alpha = alpha
        self.steps = steps

    def attack(self, x, labels, model, hog_extractor, resnet_extractor, loss_fn, device):
        """
        Generate adversarial examples using PGD.
        
        Args:
            x: Input tensor (B, C, H, W)
            labels: Target labels (B,)
            model: Model to attack
            hog_extractor: HOG feature extractor
            resnet_extractor: ResNet feature extractor
            loss_fn: Loss function
            device: Device (cpu/cuda)
        
        Returns:
            x_adv: Adversarial examples
        """
        x_adv = x.clone().detach()
        x_adv.requires_grad = True
        
        # Random start
        delta = torch.empty_like(x).uniform_(-self.epsilon, self.epsilon).to(device)
        x_adv = torch.clamp(x + delta, 0, 1).detach()
        x_adv.requires_grad = True
        
        for step in range(self.steps):
            x_adv.grad = None
            
            with torch.enable_grad():
                hog_feat = hog_extractor.extract(x_adv)
                hog_feat = hog_feat.to(device)
                resnet_feat = resnet_extractor(x_adv)
                
                outputs = model(hog_feat, resnet_feat)
                loss = loss_fn(outputs, labels)
                
                loss.backward()
            
            with torch.no_grad():
                x_adv.data = x_adv.data + self.alpha * x_adv.grad.sign()
                
                delta = x_adv - x
                delta = torch.clamp(delta, -self.epsilon, self.epsilon)
                x_adv.data = torch.clamp(x + delta, 0, 1)
        
        return x_adv.detach()

pgd_attack = PGDAttack(
    epsilon=CONFIG['pgd_epsilon'],
    alpha=CONFIG['pgd_alpha'],
    steps=CONFIG['pgd_steps']
)

print("✓ PGD Attack initialized")
print(f"  - Epsilon: {CONFIG['pgd_epsilon']}")
print(f"  - Alpha: {CONFIG['pgd_alpha']}")
print(f"  - Steps: {CONFIG['pgd_steps']}")

## CELL 10: Compute Feature Dimensions

In [ ]:
print("🔧 Recalculating feature dimensions...\n")

images, labels_batch = next(iter(train_loader))
images = images.to(CONFIG['device'])

hog_features = hog_extractor.extract(images)
print(f"HOG features shape: {hog_features.shape}")
hog_dim = hog_features.shape[1]

resnet_features = resnet_extractor(images)
print(f"ResNet features shape: {resnet_features.shape}")
resnet_dim = resnet_features.shape[1]

total_dim = hog_dim + resnet_dim
print(f"\nTotal fused features: {total_dim}-dim")
print(f"  - HOG: {hog_dim}-dim")
print(f"  - ResNet50: {resnet_dim}-dim\n")

CONFIG['hog_features_dim'] = hog_dim
CONFIG['resnet_features_dim'] = resnet_dim
CONFIG['fused_dim'] = total_dim

## CELL 11: AFFETDS Model - Feature Fusion

In [ ]:
class AFFFETDSModel(nn.Module):
    """Adversarial Feature Fusion Ensemble for Tumor Deepfake Detection"""
    def __init__(self, hog_dim, resnet_dim, num_classes=2):
        super(AFFFETDSModel, self).__init__()

        fused_dim = hog_dim + resnet_dim

        self.fusion = nn.Sequential(
            nn.Linear(fused_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.2),
        )

        self.classifier = nn.Linear(256, num_classes)

    def forward(self, hog_feat, resnet_feat):
        """
        Args:
            hog_feat: (B, hog_dim)
            resnet_feat: (B, resnet_dim)
        Returns:
            outputs: (B, num_classes)
        """
        fused = torch.cat([hog_feat, resnet_feat], dim=1)
        hidden = self.fusion(fused)
        outputs = self.classifier(hidden)
        return outputs

model = AFFFETDSModel(
    hog_dim=CONFIG['hog_features_dim'],
    resnet_dim=CONFIG['resnet_features_dim'],
    num_classes=2
).to(CONFIG['device'])

print(f"✓ AFFETDS Model created")
print(f"  - Input: {CONFIG['fused_dim']}-dim fused features")
print(f"  - Hidden: 1024 → 512 → 256")
print(f"  - Output: 2 classes")

optimizer = optim.Adam(model.parameters(), lr=CONFIG['lr'])
loss_fn = nn.CrossEntropyLoss()
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

print(f"✓ Training setup complete")

## CELL 12: Training Loop with Adversarial Training

In [ ]:
from torch.cuda.amp import GradScaler, autocast
import time

scaler = GradScaler()

history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'epoch_time': []
}

best_val_acc = 0

checkpoint_dir = CONFIG['checkpoint_dir']
os.makedirs(checkpoint_dir, exist_ok=True)
best_model_path = f'{checkpoint_dir}/best_affetds_model.pt'

print(f"✓ Checkpoint directory created at: {checkpoint_dir}")
print(f"✓ Best model will be saved to: {best_model_path}\n")

patience = 5
patience_counter = 0

if CONFIG['is_checking_accuracy']:
    print(f"🔍 DEBUG MODE ENABLED: Training on 10% of data for quick testing")
    train_loader_list = list(train_loader)
    val_loader_list = list(val_loader)

    debug_train_loader = train_loader_list[:max(1, len(train_loader_list) // 10)]
    debug_val_loader = val_loader_list[:max(1, len(val_loader_list) // 10)]

    train_size_used = sum(len(batch[0]) for batch in debug_train_loader)
    val_size_used = sum(len(batch[0]) for batch in debug_val_loader)

    print(f"   - Train: {len(debug_train_loader)} batches (~{train_size_used} images from {len(train_dataset)})")
    print(f"   - Val:   {len(debug_val_loader)} batches (~{val_size_used} images from {len(val_dataset)})")
    print(f"   ⏱️   Expected epoch time: ~10-15% of normal\n")
    active_train_loader = debug_train_loader
    active_val_loader = debug_val_loader
else:
    print(f"🚀 FULL TRAINING MODE: Using complete dataset\n")
    active_train_loader = train_loader
    active_val_loader = val_loader

print(f"🚀 Training AFFETDS for {CONFIG['epochs']} epochs (OPTIMIZED WITH ADVERSARIAL)\n")
print(f"Configuration:")
print(f"  - Batch size: {CONFIG['batch_size']}")
print(f"  - Image size: {CONFIG['img_size']}×{CONFIG['img_size']}")
print(f"  - Fused features: {CONFIG['fused_dim']}-dim (HOG + ResNet50)")
print(f"  - Mixed Precision: ENABLED (FP16)")
print(f"  - Adversarial Training: ENABLED (50% PGD attacks per batch)")
print(f"  - DEBUG MODE: {'ON (10% data)' if CONFIG['is_checking_accuracy'] else 'OFF (full data)'}")
print(f"  - Expected time: {'2-3 min/epoch' if CONFIG['is_checking_accuracy'] else '25-35 min/epoch'}\n")
print("="*80 + "\n")

start_training = time.time()

for epoch in range(CONFIG['epochs']):
    epoch_start = time.time()

    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    batch_count = 0
    adv_count = 0

    for batch_idx, (images, labels) in enumerate(tqdm(active_train_loader, desc=f"Epoch {epoch+1} Train")):
        images = images.to(CONFIG['device'], non_blocking=True)
        labels = labels.to(CONFIG['device'], non_blocking=True)

        with torch.enable_grad():
            images_adv = pgd_attack.attack(images, labels, model, hog_extractor, resnet_extractor, loss_fn, CONFIG['device'])

        mix_idx = torch.rand(len(images)) < 0.5
        images_mixed = images.clone()
        images_mixed[mix_idx] = images_adv[mix_idx]
        adv_count += mix_idx.sum().item()

        optimizer.zero_grad()

        with autocast():
            hog_feat = hog_extractor.extract(images_mixed)
            hog_feat = hog_feat.to(CONFIG['device'])
            resnet_feat = resnet_extractor(images_mixed)
            outputs = model(hog_feat, resnet_feat)
            loss = loss_fn(outputs, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        train_loss += loss.item()
        _, predicted = outputs.max(1)
        train_correct += predicted.eq(labels).sum().item()
        train_total += labels.size(0)
        batch_count += 1

    train_loss /= batch_count
    train_acc = train_correct / train_total
    adv_pct = (adv_count / max(train_total, 1)) * 100

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    val_batches = 0

    with torch.no_grad():
        for images, labels in active_val_loader:
            images = images.to(CONFIG['device'], non_blocking=True)
            labels = labels.to(CONFIG['device'], non_blocking=True)

            with autocast():
                hog_feat = hog_extractor.extract(images)
                hog_feat = hog_feat.to(CONFIG['device'])
                resnet_feat = resnet_extractor(images)
                outputs = model(hog_feat, resnet_feat)
                loss = loss_fn(outputs, labels)

            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_correct += predicted.eq(labels).sum().item()
            val_total += labels.size(0)
            val_batches += 1

    val_loss /= val_batches
    val_acc = val_correct / val_total

    epoch_time = time.time() - epoch_start

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['epoch_time'].append(epoch_time)

    scheduler.step(val_loss)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        best_indicator = "✅"
        patience_counter = 0
    else:
        best_indicator = "  "
        patience_counter += 1

    elapsed = sum(history['epoch_time'])
    avg_time = elapsed / (epoch + 1)
    remaining_epochs = CONFIG['epochs'] - epoch - 1
    eta_hours = (avg_time * remaining_epochs) / 3600

    print(f"{best_indicator} Epoch {epoch+1:2d}/{CONFIG['epochs']} | "
          f"T.Loss: {train_loss:.4f} | T.Acc: {train_acc:.4f} ({adv_pct:.0f}% adv) | "
          f"V.Loss: {val_loss:.4f} | V.Acc: {val_acc:.4f} | "
          f"Time: {epoch_time:.0f}s | ETA: {eta_hours:.1f}h")

    if patience_counter >= patience:
        print(f"\n⏹️  Early stopping triggered")
        break

total_time = time.time() - start_training
print("\n" + "="*80)
print(f"✅ Training completed!")
print(f"   Best validation accuracy: {best_val_acc:.4f}")
print(f"   Total time: {total_time/3600:.2f} hours")
print(f"   Average time per epoch: {sum(history['epoch_time'])/len(history['epoch_time']):.0f}s")
print("="*80)

model.load_state_dict(torch.load(best_model_path))

## CELL 13: Evaluate on Test Set

In [ ]:
model.load_state_dict(torch.load(best_model_path))
print("✓ Best model loaded")

model.eval()
all_preds = []
all_labels = []
all_probs = []

print(f"\n📊 TESTING ON {len(test_dataset)} Test Samples...")

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images = images.to(CONFIG['device'], non_blocking=True)
        labels = labels.to(CONFIG['device'], non_blocking=True)

        hog_feat = hog_extractor.extract(images)
        hog_feat = hog_feat.to(CONFIG['device'])

        resnet_feat = resnet_extractor(images)

        outputs = model(hog_feat, resnet_feat)
        probs = torch.softmax(outputs, dim=1)
        _, pred = outputs.max(1)

        all_preds.extend(pred.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

test_acc = accuracy_score(all_labels, all_preds)
test_prec = precision_score(all_labels, all_preds, zero_division=0)
test_rec = recall_score(all_labels, all_preds, zero_division=0)
test_f1 = f1_score(all_labels, all_preds, zero_division=0)
test_auc = roc_auc_score(all_labels, all_probs[:, 1])

print(f"\n📊 TEST SET EVALUATION:")
print(f"="*50)
print(f"Accuracy:  {test_acc:.4f}")
print(f"Precision: {test_prec:.4f}")
print(f"Recall:    {test_rec:.4f}")
print(f"F1-Score:  {test_f1:.4f}")
print(f"AUC-ROC:   {test_auc:.4f}")
print(f"="*50)

if test_acc > 0.96:
    print(f"\n⚠️  WARNING: Test accuracy is {test_acc:.4f}%")
    print(f"   This may indicate fake images are too simplistic")
else:
    print(f"\n✅ Test accuracy {test_acc:.4f} is realistic (target: 85-93%)")

## CELL 14: Save Model to Kaggle Working Directory

In [ ]:
print("\n" + "="*80)
print("💾 FINALIZING MODEL SAVE TO KAGGLE WORKING DIRECTORY")
print("="*80)

try:
    final_model_path = f'{CHECKPOINT_DIR}/final_affetds_model.pt'
    final_model_full_path = f'{CHECKPOINT_DIR}/final_affetds_model_full.pth'
    config_path = f'{CHECKPOINT_DIR}/config.json'

    torch.save(model.state_dict(), final_model_path)
    print(f"✓ State dict saved: {final_model_path}")

    torch.save({
        'model_state': model.state_dict(),
        'config': CONFIG,
        'best_val_acc': best_val_acc,
        'test_results': {
            'accuracy': test_acc,
            'precision': test_prec,
            'recall': test_rec,
            'f1': test_f1,
            'auc': test_auc
        }
    }, final_model_full_path)
    print(f"✓ Full checkpoint saved: {final_model_full_path}")

    with open(config_path, 'w') as f:
        json.dump(CONFIG, f, indent=2, default=str)
    print(f"✓ Configuration saved: {config_path}")

    if os.path.exists(best_model_path):
        import shutil
        backup_path = f'{CHECKPOINT_DIR}/best_affetds_model_backup.pt'
        shutil.copy(best_model_path, backup_path)
        print(f"✓ Best model backed up: {backup_path}")

    print(f"\n✅ All models successfully saved to Kaggle working directory:")
    print(f"   📁 Directory: {CHECKPOINT_DIR}")
    print(f"   ✓ Latest model: {final_model_path}")
    print(f"   ✓ Best model: {best_model_path}")
    print(f"   ✓ Configuration: {config_path}")

except Exception as e:
    print(f"\n⚠️ Error saving models: {str(e)}")
    print(f"   Please ensure /kaggle/working/ is writable")

print("="*80)

## CELL 15: Confusion Matrix & ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=True,
            xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
axes[0].set_title('Confusion Matrix (Test Set)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

fpr, tpr, _ = roc_curve(all_labels, all_probs[:, 1])
roc_auc = auc(fpr, tpr)

axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve (Test Set)', fontsize=12, fontweight='bold')
axes[1].legend(loc="lower right")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CHECKPOINT_DIR}/confusion_matrix_roc.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Confusion matrix and ROC curve saved to Kaggle working directory")

## CELL 16: Training History Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], label='Train Loss', marker='o', markersize=3)
axes[0].plot(history['val_loss'], label='Val Loss', marker='s', markersize=3)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss Curve', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history['train_acc'], label='Train Acc', marker='o', markersize=3)
axes[1].plot(history['val_acc'], label='Val Acc', marker='s', markersize=3)
if 'test_acc' in locals():
    axes[1].axhline(y=test_acc, color='r', linestyle='--', label=f'Test Acc: {test_acc:.4f}')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training Accuracy Curve', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CHECKPOINT_DIR}/training_history.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Training history visualization saved")

## CELL 17: t-SNE Feature Visualization

In [ ]:
print("📊 Extracting fused features for t-SNE visualization...")
from sklearn.manifold import TSNE

model.load_state_dict(torch.load(best_model_path))
model.eval()

all_features = []
all_labels_tsne = []

with torch.no_grad():
    for images, labels_batch in tqdm(test_loader, desc="Feature extraction"):
        images = images.to(device)
        hog_feat = hog_extractor.extract(images)
        hog_feat = hog_feat.to(device)
        resnet_feat = resnet_extractor(images)
        
        fused = torch.cat([hog_feat, resnet_feat], dim=1)
        all_features.extend(fused.cpu().numpy())
        all_labels_tsne.extend(labels_batch.numpy())

all_features = np.array(all_features)
all_labels_tsne = np.array(all_labels_tsne)

print("Applying t-SNE (this may take a moment)...")
num_samples = all_features.shape[0]
perplexity = min(30, max(5, num_samples // 3))
print(f"  Using perplexity={perplexity} for {num_samples} samples\n")

tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity, n_iter=1000)
features_2d = tsne.fit_transform(all_features)

fig, ax = plt.subplots(figsize=(10, 8))
for label_val in [0, 1]:
    mask = all_labels_tsne == label_val
    label_name = 'Real MRI' if label_val == 0 else 'Fake MRI (Tumor)'
    ax.scatter(features_2d[mask, 0], features_2d[mask, 1],
               alpha=0.6, s=50, label=label_name)

ax.set_xlabel('t-SNE Dimension 1')
ax.set_ylabel('t-SNE Dimension 2')
ax.set_title('t-SNE Visualization of Fused Features (Deep + HOG)', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{CHECKPOINT_DIR}/tsne_features.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ t-SNE visualization saved")

## CELL 18: Final Results Summary

In [ ]:
print("\n" + "="*70)
print("🎯 AFFETDS KAGGLE TRAINING COMPLETE")
print("="*70)

print("\n📊 FINAL TEST RESULTS:")
print(f"  ✓ Accuracy:  {test_acc:.4f}")
print(f"  ✓ Precision: {test_prec:.4f}")
print(f"  ✓ Recall:    {test_rec:.4f}")
print(f"  ✓ F1-Score:  {test_f1:.4f}")
print(f"  ✓ AUC-ROC:   {test_auc:.4f}")

print("\n🔧 CONFIGURATION USED:")
print(f"  ✓ Deep Features: ResNet50 (Layer4 avgpool, 2048-dim)")
print(f"  ✓ Handcrafted Features: HOG (9 orientations, 16px cells) → {CONFIG['hog_features_dim']}-dim")
print(f"  ✓ Fusion: Concatenation → {CONFIG['fused_dim']}-dim")
print(f"  ✓ Classifier: Neural network (Linear layers + BatchNorm)")
print(f"  ✓ Adversarial Training: PGD (ε={CONFIG['pgd_epsilon']}, α={CONFIG['pgd_alpha']}, {CONFIG['pgd_steps']} steps)")
print(f"  ✓ Optimizer: Adam (lr={CONFIG['lr']}, epochs={CONFIG['epochs']}, batch={CONFIG['batch_size']})")
print(f"  ✓ Environment: Kaggle GPU")

print("\n💾 OUTPUT FILES SAVED TO /kaggle/working/:")
print(f"  ✓ best_affetds_model.pt (model state dict)")
print(f"  ✓ final_affetds_model.pt (final model)")
print(f"  ✓ final_affetds_model_full.pth (model + config + metrics)")
print(f"  ✓ config.json (training configuration)")
print(f"  ✓ confusion_matrix_roc.png (evaluation plots)")
print(f"  ✓ training_history.png (training curves)")
print(f"  ✓ tsne_features.png (feature visualization)")

print("\n" + "="*70)
print(f"✅ AFFETDS KAGGLE TRAINING COMPLETE!")
print(f"   Test Accuracy: {test_acc:.1%}")
print(f"   AUC-ROC: {test_auc:.3f}")
print("="*70)